# LSTM Gesture Recognition: Data Augmentation and Model Training

This notebook demonstrates data augmentation and LSTM-based classification for 5 gesture classes (a, b, c, j, p), each with 15 samples.

## 1. Import Required Libraries
Import libraries for data handling, augmentation, and model building.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

## 2. Load and Explore Dataset
Load all per-sequence CSVs, assign labels, and display basic statistics.

In [ ]:
# Load all per-sequence CSVs and assign labels
DATASET_DIR = 'dataset'
SEQUENCE_LENGTH = 30

def load_sequences(dataset_dir):
    X, y = [], []
    for label in os.listdir(dataset_dir):
        label_dir = os.path.join(dataset_dir, label)
        if not os.path.isdir(label_dir):
            continue
        for csv_file in glob.glob(os.path.join(label_dir, '*.csv')):
            df = pd.read_csv(csv_file, header=None)
            if len(df) == SEQUENCE_LENGTH:
                X.append(df.values)
                y.append(label)
    return np.array(X), np.array(y)

X, y = load_sequences(DATASET_DIR)
print(f"Loaded {len(X)} sequences. Shape: {X.shape}")
print(f"Classes: {np.unique(y)}")
print(f"Samples per class: {pd.Series(y).value_counts().to_dict()}")

# Show a sample sequence
pd.DataFrame(X[0], columns=[f'feat_{i}' for i in range(X.shape[2])])

## 3. Data Augmentation for Each Class
Apply augmentation (noise, scaling, time warping) to increase samples per class.

In [ ]:
# Simple augmentation: add Gaussian noise, random scaling, and time shifting
def augment_sequence(seq, noise_std=0.01, scale_range=(0.9, 1.1), shift_max=2):
    # Add noise
    noisy = seq + np.random.normal(0, noise_std, seq.shape)
    # Random scaling
    scale = np.random.uniform(*scale_range)
    scaled = noisy * scale
    # Time shift
    shift = np.random.randint(-shift_max, shift_max+1)
    if shift > 0:
        shifted = np.pad(scaled, ((shift,0),(0,0)), mode='edge')[:-shift]
    elif shift < 0:
        shifted = np.pad(scaled, ((0,-shift),(0,0)), mode='edge')[-shift:]
    else:
        shifted = scaled
    return shifted

# Augment to at least 50 samples per class
def augment_data(X, y, min_samples=50):
    X_aug, y_aug = list(X), list(y)
    for label in np.unique(y):
        idx = np.where(y == label)[0]
        needed = max(0, min_samples - len(idx))
        for _ in range(needed):
            seq = X[random.choice(idx)]
            X_aug.append(augment_sequence(seq))
            y_aug.append(label)
    return np.array(X_aug), np.array(y_aug)

X_aug, y_aug = augment_data(X, y, min_samples=50)
print(f"After augmentation: {len(X_aug)} sequences. Samples per class: {pd.Series(y_aug).value_counts().to_dict()}")

## 4. Prepare Data for Training
Encode labels, split into train/validation, and reshape for LSTM.

In [ ]:
# Encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y_aug)
num_classes = len(le.classes_)

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X_aug, y_enc, test_size=0.2, stratify=y_enc, random_state=42)

# One-hot encode labels for Keras
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)

print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}")

## 5. Build LSTM Model
Define a simple LSTM-based neural network for gesture classification.

In [ ]:
# Build a simple LSTM model
model = Sequential([
    LSTM(64, input_shape=(SEQUENCE_LENGTH, X_train.shape[2]), return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 6. Train the Model
Train the LSTM model on the augmented dataset and monitor accuracy.

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=30,
    batch_size=16,
    verbose=1
)

## 7. Evaluate Model Performance
Evaluate the trained model and display accuracy, confusion matrix, and classification report.

In [ ]:
# Evaluate model
val_preds = model.predict(X_val)
val_pred_labels = np.argmax(val_preds, axis=1)

print("Classification Report:\n", classification_report(y_val, val_pred_labels, target_names=le.classes_))

cm = confusion_matrix(y_val, val_pred_labels)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()